# 02 — Generator selection under the documented amendment

Operational specification: [`docs/PROTOCOL.md`, Sections 3–5](../../docs/PROTOCOL.md#3-post-benchmark-gate-calibration-audit).

This results-only notebook selects one fine-tuned and one from-scratch generator after applying the
approved post-benchmark Option B amendment. It loads no encoder, regenerates no image, and never
accesses test data. The notebook preserves both the original zero-eligible outcome and the amended
safety-gate result. It validates each manual choice against the registry, metric completeness, image
count, test-access rule, and amended safety gates; after validation, it writes a content-aware
selection contract and evidence snapshot.

## 1. Load benchmark evidence, registry, and active amendment

The cell resolves the canonical benchmark summary and uses `generator_summary_corrected.csv` whenever
that file exists; otherwise it uses the canonical summary. It then loads the registry, protocol, active
amendment, confirmed-duplicate rates, and paired-difference table. The portable selection-evidence
snapshot is not loaded here: it is rebuilt only when the final save cell writes the selection contract.
The displayed source path and row count make the chosen summary explicit before ranking.

In [ ]:
from pathlib import Path
import json
import sys
import csv
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
from notebooks.utility.generator_benchmark import load_protocol, load_registry, rank_generator_family, save_selected_generators
protocol = load_protocol(ROOT)
registry = load_registry(ROOT)
canonical_metrics_path = ROOT / protocol['outputs']['metrics']
corrected_metrics_path = canonical_metrics_path.with_name('generator_summary_corrected.csv')
metrics_path = corrected_metrics_path if corrected_metrics_path.is_file() else canonical_metrics_path
benchmark_rows = list(csv.DictReader(metrics_path.open())) if metrics_path.is_file() else []
paired_path = ROOT / protocol['outputs']['paired_differences']
paired_rows = list(csv.DictReader(paired_path.open())) if paired_path.is_file() else []
{'benchmark_summary_source': str(metrics_path.relative_to(ROOT)),
 'n_benchmark_rows': len(benchmark_rows)} if benchmark_rows else 'Not yet evaluated'


## 2. Compare the original gates with Option B safety eligibility

The original preregistered gates and the Option B amendment are evaluated against the effective fields
in the selected benchmark summary. Option B makes coverage and pHash-only similarity descriptive while
retaining technical validity, exact/confirmed duplication, train memorization, effective
provenance/lineage fields, and test access as blocking inputs. Under the local-data-trust benchmark
rerun, historical fingerprint mismatches remain separate diagnostics; this selection cell does not
recompute them.

In [ ]:
filtered_rows = [row for row in benchmark_rows if row.get('condition') == 'FILTERED']
gates = protocol['eligibility_gates']
finetuned_ranking = rank_generator_family(filtered_rows, 'finetuned', gates) if filtered_rows else []
fromscratch_ranking = rank_generator_family(filtered_rows, 'from_scratch', gates) if filtered_rows else []
outcome = {'eligible': sum(bool(row['eligible']) for row in finetuned_ranking + fromscratch_ranking),
           'exclusions': [(row['generator_id'], row['exclusion_reasons']) for row in finetuned_ranking + fromscratch_ranking],
           'finetuned_rank': [(row['generator_id'], row['family_rank']) for row in finetuned_ranking],
           'from_scratch_rank': [(row['generator_id'], row['family_rank']) for row in fromscratch_ranking]}
outcome


## 3. Display descriptive evidence and the unchanged KID-primary hierarchy

The tables expose fidelity, coverage, stability, duplication, memorization, effective
provenance/lineage, and efficiency fields for candidates present in the selected summary. They do not
recompute metrics or display the fresh benchmark's `*_recorded` provenance diagnostics unless those
columns are explicitly joined. Within the amended eligible set, ordering remains RAD-DINO-KID-primary
with the registered deterministic tie-breaks; non-selectable rows present in the summary remain visible.

In [ ]:
display_columns = ['generator_id', 'family_rank', 'raddino_kid', 'raddino_kid_stability_low', 'raddino_kid_stability_high',
                   'raddino_coverage', 'raddino_precision', 'raddino_fid', 'inception_kid', 'raddino_kid_std',
                   'perceptual_hash_duplicate_rate',  # descriptive only, not a gate
                   'train_memorization_rate', 'synthetic_exact_duplicate_rate',
                   'generation_seconds_per_image', 'efficiency_status']
[[{column: row.get(column) for column in display_columns} for row in ranking] for ranking in (finetuned_ranking, fromscratch_ranking)]


## 4. Review paired differences and declare the two manual choices

Paired repeated-subsampling differences describe how candidate KID estimates move under the shared
sampling plan. The proposed top-ranked candidates are displayed beside the two explicit manual
constants. Final validation checks family membership, registry selection eligibility, metric
completeness, the 1,361-image requirement, test access, and amended safety gates. It does not require a
manual choice to equal the proposed top-ranked row; the content-aware evidence snapshot and selection
contract are created only after that validation succeeds.

In [ ]:
SELECTED_FINETUNED_GENERATOR = "02_sd21_filtered_100steps"
SELECTED_FROM_SCRATCH_GENERATOR = "07_ldm_sdvae_extra1361"
PROPOSED_FINETUNED_GENERATOR = next((row['generator_id'] for row in finetuned_ranking if row['eligible']), None)
PROPOSED_FROM_SCRATCH_GENERATOR = next((row['generator_id'] for row in fromscratch_ranking if row['eligible']), None)
SELECTION_NOTES = ('Eligibility uses the technical/scientific safety gates only (image count, exact-duplicate and '
                   'train-memorization rates, corruption, metric completeness, test isolation, registry role); '
                   'perceptual-hash rate and RAD-DINO coverage are descriptive ranking metrics. The preregistered '
                   'RAD-DINO KID-primary hierarchy selects G02 (fine-tuned) and G07 (from-scratch).')
{'selected': (SELECTED_FINETUNED_GENERATOR, SELECTED_FROM_SCRATCH_GENERATOR),
 'proposed_top_rank': (PROPOSED_FINETUNED_GENERATOR, PROPOSED_FROM_SCRATCH_GENERATOR),
 'paired_generator_differences': paired_rows}


## 5. Validate and persist the downstream selection contract

With `SAVE_SELECTION=True`, the two validated choices and amendment notes are written atomically to
the canonical selection file. The saved decision binds generator IDs to content-hashed benchmark,
provenance, amendment, and manifest evidence so downstream classifiers cannot silently consume a
different synthetic pool. Setting the flag to `False` provides a read-only review without modifying
the contract.

In [ ]:
SAVE_SELECTION = True
if SAVE_SELECTION and benchmark_rows:
    output = save_selected_generators(ROOT, SELECTED_FINETUNED_GENERATOR, SELECTED_FROM_SCRATCH_GENERATOR,
                                      benchmark_rows, notes=SELECTION_NOTES)
    print('Saved selection to', output)
    print(json.dumps(json.loads(Path(output).read_text()), indent=1))
else:
    print('Selection not saved. Requires benchmark results.')
